# Medical Decision Support Expert System — Visual Test Suite
This notebook imports `RuleAlzheimer`, `questions`, `answer`, `contradictions`, `goals`, and `InferenceEngine` directly from your existing `.py` modules and runs **6 visual end-to-end test scenarios** comparing **Expected vs. Actual** outputs.

> **Setup Note:** Update the module names in **Cell 1** (`KB_MODULE` and `ENGINE_MODULE`) to match your exact `.py` filenames (without the `.py` extension) if they differ.

In [4]:
# ============================================================================
# CELL 1: Direct Imports from Your Existing .py Files
# ============================================================================
import copy
import importlib
from unittest.mock import patch
from IPython.display import display, HTML
import matplotlib.pyplot as plt

# Change these two strings to match your exact .py filenames (without .py)
KB_MODULE = "alzheimer"
ENGINE_MODULE = "inferenceengine"

kb = importlib.import_module(KB_MODULE)
eng = importlib.import_module(ENGINE_MODULE)

RuleAlzheimer = kb.RuleAlzheimer
questions = kb.questions
answer = kb.answer
contradictions = kb.contradictions
goals = kb.goals
InferenceEngine = eng.InferenceEngine

print(f"Successfully imported Knowledge Base from '{KB_MODULE}.py' and Inference Engine from '{ENGINE_MODULE}.py'.")

Successfully imported Knowledge Base from 'alzheimer.py' and Inference Engine from 'inferenceengine.py'.


In [5]:
# ============================================================================
# CELL 2: Visual Test Runner & Expected vs. Actual Comparison Helper
# ============================================================================
test_summary_records = []

def run_visual_test(
    test_name: str,
    description: str,
    mode: str,
    simulated_inputs: list,
    expected: dict,
    run_explain: bool = True
):
    """Runs an automated scenario with mocked user inputs and renders Expected vs. Actual cards."""
    engine = InferenceEngine(
        questions=questions,
        answers=copy.deepcopy(answer),
        rule_class=RuleAlzheimer,
        contradictions=contradictions,
        goals=goals
    )

    print("=" * 88)
    print(f"RUNNING: {test_name}")
    print("=" * 88)

    with patch("builtins.input", side_effect=simulated_inputs):
        if mode == "forward":
            engine.forward_chaining()
        elif mode == "backward":
            engine.backward_chaining()
        else:
            raise ValueError("mode must be 'forward' or 'backward'")

    if run_explain:
        print("\n--- Rule Audit Explanation ---")
        engine.explain_rules()

    # Gather actual state
    triggered = engine.kb.print_rules()
    actual_conclusion = triggered[0][0] if triggered else "None"
    actual_confidence = triggered[0][1] if triggered else 0.0
    actual_yes = sum(1 for v in engine.answers.values() if v == "yes")
    actual_no = sum(1 for v in engine.answers.values() if v == "no")
    actual_dk = sum(1 for v in engine.answers.values() if v == "don't know")
    actual_unresolved = len(engine.unresolved_contradictions)

    actual = {
        "conclusion": actual_conclusion,
        "confidence": actual_confidence,
        "yes_count": actual_yes,
        "no_count": actual_no,
        "dk_count": actual_dk,
        "unresolved_contradictions": actual_unresolved,
    }

    checks = [
        ("Conclusion", expected["conclusion"], actual["conclusion"]),
        ("Confidence Score", expected["confidence"], actual["confidence"]),
        ("Confirmed 'yes' Signs", expected["yes_count"], actual["yes_count"]),
        ("Confirmed 'no' Signs", expected["no_count"], actual["no_count"]),
        ("Unknown ('don't know')", expected["dk_count"], actual["dk_count"]),
        ("Unresolved Contradictions", expected["unresolved_contradictions"], actual["unresolved_contradictions"]),
    ]

    all_passed = all(exp == act for _, exp, act in checks)
    badge_color = "#16a34a" if all_passed else "#dc2626"
    badge_text = "PASS" if all_passed else "FAIL"

    rows_html = ""
    for label, exp_val, act_val in checks:
        row_ok = (exp_val == act_val)
        status_icon = "✅" if row_ok else "❌"
        bg = "#f0fdf4" if row_ok else "#fef2f2"
        rows_html += f"""
        <tr style="background:{bg};">
            <td style="padding:8px;border:1px solid #cbd5e1;font-weight:600;">{label}</td>
            <td style="padding:8px;border:1px solid #cbd5e1;">{exp_val}</td>
            <td style="padding:8px;border:1px solid #cbd5e1;">{act_val}</td>
            <td style="padding:8px;border:1px solid #cbd5e1;text-align:center;">{status_icon}</td>
        </tr>
        """

    card_html = f"""
    <div style="border:1px solid #94a3b8;border-radius:8px;padding:16px;margin:14px 0;font-family:sans-serif;background:#ffffff;">
        <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
            <h3 style="margin:0;color:#0f172a;">{test_name}</h3>
            <span style="background:{badge_color};color:#fff;padding:4px 12px;border-radius:12px;font-weight:bold;">{badge_text}</span>
        </div>
        <p style="color:#475569;margin-top:0;">{description}</p>
        <table style="width:100%;border-collapse:collapse;font-size:14px;color:#0f172a;">
            <thead>
                <tr style="background:#e2e8f0;">
                    <th style="padding:8px;border:1px solid #cbd5e1;text-align:left;">Metric</th>
                    <th style="padding:8px;border:1px solid #cbd5e1;text-align:left;">Expected</th>
                    <th style="padding:8px;border:1px solid #cbd5e1;text-align:left;">Actual</th>
                    <th style="padding:8px;border:1px solid #cbd5e1;text-align:center;">Match</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
    </div>
    """
    display(HTML(card_html))

    test_summary_records.append({
        "name": test_name,
        "passed": all_passed,
        "confidence": actual_confidence,
        "yes": actual_yes,
        "no": actual_no,
        "dk": actual_dk
    })

In [8]:
# ============================================================================
# TEST 1: Healthy Baseline (Forward Chaining — 0 Warning Signs)
# ============================================================================
run_visual_test(
    test_name="Test 1: Healthy Baseline (0 Signs)",
    description="User answers 'no' to all 10 warning signs. Expects 'No warning signs reported' with 0.0 confidence.",
    mode="forward",
    simulated_inputs=["no"] * 10,
    expected={
        "conclusion": "No warning signs reported",
        "confidence": 0.0,
        "yes_count": 0,
        "no_count": 10,
        "dk_count": 0,
        "unresolved_contradictions": 0,
    }
)

RUNNING: Test 1: Healthy Baseline (0 Signs)
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+------------------+--------------------+------------------------+-------------------------+-----------------+----------------------------------+
| Conclusion       |   Confidence_score | Recommendation         | Rationale               | Source          | Rule as text                     |
+==================+====================+========================+=========================+=================+==================================+
| No warning signs |                  0 | No warning signs were  | None of the ten warning | Team assumption | IFall warning signs = no THEN no |
| reported         |                    | reported. Seek         | signs were reported.    |                 | warning signs reported           |
|                  |                    | professional advice if |                         |          

Metric,Expected,Actual,Match
Conclusion,No warning signs reported,No warning signs reported,✅
Confidence Score,0.0,0.0,✅
Confirmed 'yes' Signs,0,0,✅
Confirmed 'no' Signs,10,10,✅
Unknown ('don't know'),0,0,✅
Unresolved Contradictions,0,0,✅


In [9]:
# ============================================================================
# TEST 2: Single Base Rule Trigger (Forward Chaining — 1 Warning Sign)
# ============================================================================
run_visual_test(
    test_name="Test 2: Single Base Rule (Language Sign Only)",
    description="User answers 'yes' only to 'language' (Q4). Expects Base Rule 8 ('Possible communication concern', confidence 0.40).",
    mode="forward",
    simulated_inputs=[
        "no",   # memory_loss
        "no",   # confusion_time_place
        "no",   # familiar_tasks
        "yes",  # language
        "no", "no", "no", "no", "no", "no"
    ],
    expected={
        "conclusion": "Possible communication concern",
        "confidence": 0.4,
        "yes_count": 1,
        "no_count": 9,
        "dk_count": 0,
        "unresolved_contradictions": 0,
    }
)

RUNNING: Test 2: Single Base Rule (Language Sign Only)
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+------------------------+--------------------+----------------------+-----------------------------+-----------------+---------------------------------+
| Conclusion             |   Confidence_score | Recommendation       | Rationale                   | Source          | Rule as text                    |
+========================+====================+======================+=============================+=================+=================================+
| Possible communication |                0.4 | Consult a healthcare | Difficulty finding words or | Team assumption | IF language = yes THEN Possible |
| concern                |                    | professional.        | following conversations may |                 | communication concern           |
|                        |                    |         

Metric,Expected,Actual,Match
Conclusion,Possible communication concern,Possible communication concern,✅
Confidence Score,0.4,0.4,✅
Confirmed 'yes' Signs,1,1,✅
Confirmed 'no' Signs,9,9,✅
Unknown ('don't know'),0,0,✅
Unresolved Contradictions,0,0,✅


In [10]:
# ============================================================================
# TEST 3: Input Validation, Contradiction Resolution & Follow-Up Questions
# ============================================================================
run_visual_test(
    test_name="Test 3: Contradiction Review + Follow-Up Clarification (3 Signs)",
    description="Tests 'example' and invalid input prompts, resolves (memory_loss, misplacing) contradiction, clarifies 1 of 2 'don't know' follow-ups to 'yes' (3 total signs -> 0.70 * 9/10 = 0.63 confidence).",
    mode="forward",
    simulated_inputs=[
        "example",     # Q1: memory_loss -> prints examples
        "maybe",       # Q1: invalid input -> prints error
        "no",          # Q1: memory_loss = no
        "don't know",  # Q2: confusion_time_place = don't know
        "no",          # Q3: familiar_tasks = no
        "no",          # Q4: language = no
        "yes",         # Q5: misplacing = yes (triggers contradiction with memory_loss!)
        "don't know",  # Q6: judgment = don't know
        "no", "no", "no", "no",  # Q7-Q10 = no
        # Contradiction review prompt for (memory_loss, misplacing):
        "yes",         # Review answers? -> yes
        "yes",         # Re-answer memory_loss -> updated to yes
        "yes",         # Re-answer misplacing -> kept as yes
        # Follow-up prompts for the two 'don't know' answers:
        "yes",         # Follow-up confusion_time_place -> updated to yes (now 3 'yes' signs!)
        "no"           # Follow-up judgment -> remains don't know
    ],
    expected={
        "conclusion": "Possible early Alzheimer's indicators",
        "confidence": 0.63,
        "yes_count": 3,
        "no_count": 6,
        "dk_count": 1,
        "unresolved_contradictions": 0,
    }
)

RUNNING: Test 3: Contradiction Review + Follow-Up Clarification (3 Signs)
['Forgetting recently learned information', 'Repeatedly asking the same question', 'Forgetting important dates or events', 'Repeating statements']
Answer must be either 'yes','no' or 'don't know'. Type 'example' to see some sample cases
Possible Inconsistence detected between  memory_loss  and  misplacing

Follow-up question for Confusion Time Place
Answer updated to yes.

Follow-up question for Judgment
No change made. Answer remains don't know.
Rule-Based Expert System for Medical Decision Support.
All rules are illustrative and are not intended for actual diagnosis
+------------------------+--------------------+----------------------+-----------------------------+-----------------+----------------------------------+
| Conclusion             |   Confidence_score | Recommendation       | Rationale                   | Source          | Rule as text                     |
+========================+=================

Metric,Expected,Actual,Match
Conclusion,Possible early Alzheimer's indicators,Possible early Alzheimer's indicators,✅
Confidence Score,0.63,0.63,✅
Confirmed 'yes' Signs,3,3,✅
Confirmed 'no' Signs,6,6,✅
Unknown ('don't know'),1,1,✅
Unresolved Contradictions,0,0,✅


In [ ]:
# ============================================================================
# TEST 4: Unresolved Contradiction Flagging (5 Warning Signs)
# ============================================================================
run_visual_test(
    test_name="Test 4: Unresolved Contradiction Warning (5 Signs)",
    description="User answers 'yes' to 5 signs including familiar_tasks='yes' and planning='no', then declines review ('no'). Expects Moderate warning signs (0.80) + 1 unresolved contradiction.",
    mode="forward",
    simulated_inputs=[
        "yes",  # memory_loss
        "yes",  # confusion_time_place
        "yes",  # familiar_tasks (contradicts planning='no')
        "yes",  # language
        "yes",  # misplacing
        "no",   # judgment
        "no",   # mood_personality
        "no",   # withdrawal
        "no",   # planning
        "no",   # visual_spatial
        # Contradiction review prompt for (planning, familiar_tasks):
        "no"    # Decline review -> recorded in unresolved_contradictions
    ],
    expected={
        "conclusion": "Moderate Alzheimer's warning signs",
        "confidence": 0.8,
        "yes_count": 5,
        "no_count": 5,
        "dk_count": 0,
        "unresolved_contradictions": 1,
    }
)

In [ ]:
# ============================================================================
# TEST 5: Backward Chaining — Early Goal Satisfaction
# ============================================================================
run_visual_test(
    test_name="Test 5: Backward Chaining — Early Success (2-Sign Hypothesis)",
    description="Selects Goal 1 ('Possible cognitive decline', requires 2 'yes'). Answers 'yes' to first 2 questions -> stops immediately and auto-fills remaining 8 as 'don't know' (0.60 * 2/10 = 0.12 confidence).",
    mode="backward",
    simulated_inputs=[
        "1",    # Select Goal 1 (threshold = 2)
        "yes",  # Q1: memory_loss (1/2)
        "yes"   # Q2: confusion_time_place (2/2) -> Early exit!
    ],
    expected={
        "conclusion": "Possible cognitive decline",
        "confidence": 0.12,
        "yes_count": 2,
        "no_count": 0,
        "dk_count": 8,
        "unresolved_contradictions": 0,
    }
)

In [ ]:
# ============================================================================
# TEST 6: Backward Chaining — Early Impossibility & Rule Audit
# ============================================================================
run_visual_test(
    test_name="Test 6: Backward Chaining — Early Failure & Rule Audit (6-Sign Hypothesis)",
    description="Selects Goal 4 ('Strong Alzheimer's warning signs', requires 6 'yes'). Answers 'no' 5 times -> impossible to reach 6, stops at Q5, auto-fills remaining 5 as 'don't know', and runs explain_rules().",
    mode="backward",
    simulated_inputs=[
        "4",                      # Select Goal 4 (threshold = 6)
        "no", "no", "no", "no", "no"  # 5 'no' answers -> max possible is 5 (< 6) -> Early exit!
    ],
    expected={
        "conclusion": "No warning signs reported",
        "confidence": 0.0,
        "yes_count": 0,
        "no_count": 5,
        "dk_count": 5,
        "unresolved_contradictions": 0,
    }
)

In [ ]:
# ============================================================================
# CELL 9: Visual Dashboard Across All 6 Test Scenarios
# ============================================================================
if test_summary_records:
    labels = [f"Test {i+1}" for i in range(len(test_summary_records))]
    yes_vals = [r["yes"] for r in test_summary_records]
    no_vals = [r["no"] for r in test_summary_records]
    dk_vals = [r["dk"] for r in test_summary_records]
    conf_vals = [r["confidence"] for r in test_summary_records]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    # Stacked Bar Chart of Answer States
    ax1.bar(labels, yes_vals, label="Yes", color="#16a34a")
    ax1.bar(labels, no_vals, bottom=yes_vals, label="No", color="#dc2626")
    bottom_dk = [y + n for y, n in zip(yes_vals, no_vals)]
    ax1.bar(labels, dk_vals, bottom=bottom_dk, label="Don't Know / Auto-Filled", color="#f59e0b")
    ax1.set_title("Answer Distribution per Test Scenario", fontweight="bold")
    ax1.set_ylabel("Number of Questions (Total = 10)")
    ax1.set_ylim(0, 11.5)
    ax1.legend(loc="upper right")

    # Confidence Score Comparison
    bars = ax2.bar(labels, conf_vals, color="#2563eb")
    ax2.set_title("Resulting Confidence Score per Test Scenario", fontweight="bold")
    ax2.set_ylabel("Confidence Score (0.0 - 1.0)")
    ax2.set_ylim(0, 1.05)
    for bar, val in zip(bars, conf_vals):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.03, f"{val:.2f}", ha="center", fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("Run the test cells above first to generate the visual summary.")